# Event streaming
A higher-level stream that hands you typed projections — messages, reasoning, tool calls, state — so you consume each data channel independently instead of parsing raw tuples.

## Concept
Call agent.stream_events(..., version="v3") and iterate one of its projections. Each surfaces a different slice of the run, and stream.output holds the final state once iteration ends.

- stream.messages — model output; iterate message.text for deltas.
- stream.tool_calls — tool execution lifecycle and outputs.
- stream.values / stream.output — state snapshots / final state.

In [10]:
from pydantic import SecretStr
import os

from langchain.agents import create_agent
from langchain.tools import tool
from langchain.chat_models import init_chat_model

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"It's always sunny in {city}!"

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

agent = create_agent(
    model=model,
    tools=[get_weather]
)
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    version="v3",
)
for message in stream.messages:
    for delta in message.text:
        print(delta, end="", flush=True)

final_state = stream.output # available after the loop

The current conditions in San Francisco are mild and partly sunny, with temperatures in the mid‑60s °F (around 18 °C) and a light breeze. As usual for the Bay Area, there may be some fog rolling in, especially near the coast. If you need a more precise forecast for the next few hours, let me know and I can look it up for you!

## Watch the tool-call lifecycle — finalized calls and streamed output:

In [11]:
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Weather in SF?"}]},
    version="v3",
)

for call in stream.tool_calls:
    print(f"{call.tool_name}({call.input})")
    print(call.output, call.error)

get_weather({'city': 'San Francisco'})
None None
